In [ ]:
# SINGLE CONFIGURATION BLOCK — edit only this cell before a run.
RUN_MODE = "full"  # smoke or full
OUTPUT_DIR = "/content/counterfactual_faithfulness_stage6"
SEED = 811
MODEL_NAME = [
    "dino_wm_pusht",
    "jepa_wm_pusht",
    "dino_wm_wall",
    "jepa_wm_wall",
]
ENVIRONMENT = ["PushT", "Wall"]
HORIZONS = [1, 3, 6]
NUM_STATES = 36  # per environment in smoke mode
ACTIONS_PER_STATE = 10

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage6"
REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
FRAMESKIP = 5
FEATURE_POOL_GRID = 4
TASKS_PER_ENVIRONMENT = 12
TASK_SPLIT_COUNTS = [6, 3, 0, 3]
EVALUATION_SEEDS = [811, 1231, 1699]
PROBE_SEEDS = [6101]
BOOTSTRAP_REPS = 300
RANKING_TIE = 1e-9

ADAPTER_BOTTLENECK_DIM = 128
ADAPTER_HIDDEN_DIM = 192
ADAPTER_IMPLEMENTATION_ID = "set_noop_effect_rank_v1"
TRAINING_EPOCHS = 12
SELECTION_EPOCHS = [6, 9, 12]
TRAINING_BATCH_UNITS = 6
TRAINING_LR = 3e-4
TRAINING_WEIGHT_DECAY = 1e-4
EFFECT_WEIGHT = 1.0
RANK_WEIGHT = 0.5
ACTION_DECODE_WEIGHT = 0.1
RANK_TEMPERATURE = 0.05
POSE_ERROR_RATIO_MARGIN = 1.10
DOWNLOAD_RESULTS = True
EVIDENCE_STATUS = "EXPLORATORY_DEVELOPMENT"
TASK_FAMILY_ID = "stage5_tasks_reused_for_stage6_development"
METHODS = [
    "endpoint_only",
    "action_decode_only",
    "ranking_only",
    "counterfactual_effect_only",
    "independent_action_effect",
    "counterfactual_action_effect",
]

if RUN_MODE == "full":
    NUM_STATES = 240
    PROBE_SEEDS = [6101, 8101]
    BOOTSTRAP_REPS = 2000
    TRAINING_EPOCHS = 160
    SELECTION_EPOCHS = [80, 120, 160]
    TRAINING_BATCH_UNITS = 12
elif RUN_MODE != "smoke":
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

assert MODEL_NAME == [
    "dino_wm_pusht",
    "jepa_wm_pusht",
    "dino_wm_wall",
    "jepa_wm_wall",
]
assert ENVIRONMENT == ["PushT", "Wall"]
assert HORIZONS == [1, 3, 6]
assert ACTIONS_PER_STATE == 10
assert NUM_STATES % TASKS_PER_ENVIRONMENT == 0
assert sum(TASK_SPLIT_COUNTS) == TASKS_PER_ENVIRONMENT
assert SELECTION_EPOCHS[-1] == TRAINING_EPOCHS


# Stage 6: structured action-effect adapter development

Stage 5 validly tested a per-action readout with an endpoint-difference loss,
but the same-state objective was not better than an independent-pair control.
Stage 6 develops a stronger intervention: a learned set-aware projection that
decomposes shared world evolution from no-op-relative action effects, decodes
executed actions from centered future features, and directly optimizes physical
candidate ordering.

This notebook deliberately reuses the now-inspected Stage 5 task family. The
former final partition is renamed `development_holdout`; it is not an untouched
test set and cannot provide confirmatory paper evidence. A successful result
only nominates a frozen method for Stage 6B on numerically new tasks.

The public DINO-WM and JEPA-WM encoders and dynamics predictors remain frozen.
Only the compact action-effect adapters are trained.


In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-matched torch and torchvision builds.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")
print("No runtime restart is expected.")


In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import traceback
import zipfile
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as torch_functional
import torchvision
import yaml

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

OUT = Path(OUTPUT_DIR)
INTERMEDIATE = OUT / "intermediate"
TRUTH_ROOT = INTERMEDIATE / "truth"
MODEL_ROOT = INTERMEDIATE / "models"
LOG_DIR = OUT / "logs"
PLOT_DIR = OUT / "plots"
PROBE_DIR = OUT / "probes"
for path in [
    OUT,
    INTERMEDIATE,
    TRUTH_ROOT,
    MODEL_ROOT,
    LOG_DIR,
    PLOT_DIR,
    PROBE_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
}
print(json.dumps(VERSIONS, indent=2))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "run.log"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)
log = logging.getLogger("stage6")
log.info("Seeds set to %d; evaluation seeds=%s; readout seeds=%s", SEED, EVALUATION_SEEDS, PROBE_SEEDS)


def gpu_report(label):
    payload = {
        "label": label,
        "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
    }
    log.info("GPU memory %s", payload)
    return payload


CONFIG = {
    "RUN_MODE": RUN_MODE,
    "OUTPUT_DIR": OUTPUT_DIR,
    "SEED": SEED,
    "MODEL_NAME": MODEL_NAME,
    "ENVIRONMENT": ENVIRONMENT,
    "HORIZONS": HORIZONS,
    "NUM_STATES": NUM_STATES,
    "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "REPO_URL": REPO_URL,
    "REPO_COMMIT": REPO_COMMIT,
    "FRAMESKIP": FRAMESKIP,
    "FEATURE_POOL_GRID": FEATURE_POOL_GRID,
    "TASKS_PER_ENVIRONMENT": TASKS_PER_ENVIRONMENT,
    "TASK_SPLIT_COUNTS": TASK_SPLIT_COUNTS,
    "EVALUATION_SEEDS": EVALUATION_SEEDS,
    "PROBE_SEEDS": PROBE_SEEDS,
    "BOOTSTRAP_REPS": BOOTSTRAP_REPS,
    "RANKING_TIE": RANKING_TIE,
    "ADAPTER_BOTTLENECK_DIM": ADAPTER_BOTTLENECK_DIM,
    "ADAPTER_HIDDEN_DIM": ADAPTER_HIDDEN_DIM,
    "ADAPTER_IMPLEMENTATION_ID": ADAPTER_IMPLEMENTATION_ID,
    "TRAINING_EPOCHS": TRAINING_EPOCHS,
    "SELECTION_EPOCHS": SELECTION_EPOCHS,
    "TRAINING_BATCH_UNITS": TRAINING_BATCH_UNITS,
    "TRAINING_LR": TRAINING_LR,
    "TRAINING_WEIGHT_DECAY": TRAINING_WEIGHT_DECAY,
    "EFFECT_WEIGHT": EFFECT_WEIGHT,
    "RANK_WEIGHT": RANK_WEIGHT,
    "ACTION_DECODE_WEIGHT": ACTION_DECODE_WEIGHT,
    "RANK_TEMPERATURE": RANK_TEMPERATURE,
    "POSE_ERROR_RATIO_MARGIN": POSE_ERROR_RATIO_MARGIN,
    "DOWNLOAD_RESULTS": DOWNLOAD_RESULTS,
    "EVIDENCE_STATUS": EVIDENCE_STATUS,
    "TASK_FAMILY_ID": TASK_FAMILY_ID,
    "METHODS": METHODS,
    "pinned_dependencies": PINNED,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()
CONFIG["run_signature"] = RUN_SIGNATURE
config_path = OUT / "config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text())
    if previous.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "OUTPUT_DIR contains a different configuration; choose a new OUTPUT_DIR."
        )
config_path.write_text(json.dumps(CONFIG, indent=2) + "\n")
(OUT / "versions.json").write_text(json.dumps(VERSIONS, indent=2) + "\n")
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

PIPELINE_FAILED = False
FAILURE_MESSAGE = ""


def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)


gpu_report("startup")


In [ ]:
MODEL_BY_ENVIRONMENT = {
    "PushT": ["dino_wm_pusht", "jepa_wm_pusht"],
    "Wall": ["dino_wm_wall", "jepa_wm_wall"],
}
READOUTS = [
    "latent_distance",
    "linear_pose",
    "action_blind",
    "linear_pose_shuffled",
    "oracle_pose",
]
SPLIT_NAMES = [
    "probe_train",
    "probe_calibration",
    "regression_train",
    "final_test",
]


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=True) + "\n")


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        if not rows:
            raise ValueError(f"cannot infer fields for empty table {path}")
        fieldnames = list(rows[0])
    temporary = Path(path).with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def pair_indices(n_actions):
    return np.triu_indices(n_actions, k=1)


def pool_visual(visual, grid=FEATURE_POOL_GRID):
    value = visual.detach().float().cpu().numpy()[..., 0, :, :, :]
    height, width, dim = value.shape[-3:]
    if height % grid or width % grid:
        raise ValueError(f"feature grid {(height, width)} is not divisible by {grid}")
    fh, fw = height // grid, width // grid
    value = value.reshape(*value.shape[:-3], grid, fh, grid, fw, dim)
    value = value.mean(axis=(-4, -2))
    return value.reshape(*value.shape[:-3], -1)


def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim != 5:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}


def unit_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        raise ValueError("zero direction")
    return vector / norm


def rotate_vector(vector, degrees):
    radians = np.deg2rad(degrees)
    return np.array(
        [
            [np.cos(radians), -np.sin(radians)],
            [np.sin(radians), np.cos(radians)],
        ]
    ) @ np.asarray(vector)


def feature_metrics(truth, prediction, eps=1e-12):
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    errors = prediction - truth
    ordinary = np.sqrt(np.mean(errors**2, axis=(0, 2)))
    common = np.mean(errors, axis=0)
    centered = errors - common[None, :, :]
    action_dependent = np.sqrt(np.mean(centered**2, axis=(0, 2)))
    left, right = pair_indices(truth.shape[0])
    truth_delta = truth[left] - truth[right]
    predicted_delta = prediction[left] - prediction[right]
    pair_error = predicted_delta - truth_delta
    pair_rmse = np.sqrt(np.mean(pair_error**2, axis=(0, 2)))
    pair_scale = np.sqrt(np.mean(truth_delta**2, axis=(0, 2)))
    normalized = pair_rmse / np.maximum(pair_scale, eps)
    dot = np.sum(truth_delta * predicted_delta, axis=-1)
    denominator = (
        np.linalg.norm(truth_delta, axis=-1)
        * np.linalg.norm(predicted_delta, axis=-1)
    )
    cosine = np.divide(
        dot,
        denominator,
        out=np.zeros_like(dot),
        where=denominator > eps,
    ).mean(axis=0)
    expected_pair_mse = (
        2 * truth.shape[0] / (truth.shape[0] - 1)
    ) * action_dependent**2
    return {
        "ordinary_feature_rmse": ordinary,
        "common_mode_feature_rmse": np.sqrt(np.mean(common**2, axis=-1)),
        "action_dependent_feature_rmse": action_dependent,
        "paired_feature_rmse": pair_rmse,
        "normalized_paired_feature_rmse": normalized,
        "paired_feature_cosine": cosine,
        "pair_identity_residual": pair_rmse**2 - expected_pair_mse,
    }


def ranking_metrics(true_cost, predicted_cost, tie=RANKING_TIE):
    truth = np.asarray(true_cost, dtype=np.float64)
    prediction = np.asarray(predicted_cost, dtype=np.float64)
    selected = int(np.argmin(prediction))
    oracle = int(np.argmin(truth))
    best = float(np.min(truth))
    chosen = float(truth[selected])
    spread = float(np.max(truth) - best)
    regret = chosen - best
    normalized_regret = regret / spread if spread > tie else 0.0
    left, right = pair_indices(len(truth))
    true_margin = truth[left] - truth[right]
    predicted_margin = prediction[left] - prediction[right]
    valid = np.abs(true_margin) > tie
    credit = np.full(len(left), np.nan)
    same = np.sign(true_margin) == np.sign(predicted_margin)
    credit[valid & same] = 1.0
    credit[valid & (np.abs(predicted_margin) <= tie)] = 0.5
    credit[valid & np.isnan(credit)] = 0.0
    weights = np.abs(true_margin)
    pairwise = float(np.nanmean(credit)) if np.any(valid) else float("nan")
    weighted = (
        float(np.nansum(weights * credit) / np.sum(weights[valid]))
        if np.any(valid)
        else float("nan")
    )
    margin_scale = (
        float(np.sqrt(np.mean(true_margin[valid] ** 2))) if np.any(valid) else 0.0
    )
    normalized_margin_rmse = (
        float(
            np.sqrt(
                np.mean((predicted_margin[valid] - true_margin[valid]) ** 2)
            )
            / margin_scale
        )
        if margin_scale > tie
        else float("nan")
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": float(chosen <= best + tie),
        "regret": float(regret),
        "normalized_regret": float(normalized_regret),
        "pairwise_accuracy": pairwise,
        "weighted_pairwise_accuracy": weighted,
        "normalized_margin_rmse": normalized_margin_rmse,
        "pair_left": left,
        "pair_right": right,
        "true_margin": true_margin,
        "predicted_margin": predicted_margin,
        "pair_credit": credit,
        "pair_weight": weights,
    }


def bootstrap_mean(values, groups, repetitions, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    finite = np.isfinite(values)
    values = values[finite]
    groups = groups[finite]
    unique = np.unique(groups)
    if len(unique) == 0:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": repetitions,
        }
    grouped = [values[groups == group] for group in unique]
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions)
    for index in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        draws[index] = np.mean(
            np.concatenate([grouped[item] for item in sampled])
        )
    return {
        "estimate": float(np.mean(values)),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "n_clusters": int(len(unique)),
        "n_bootstrap": int(repetitions),
    }


def random_projection(input_dim, output_dim, seed):
    rng = np.random.default_rng(seed)
    projection = rng.standard_normal(
        (input_dim, output_dim)
    ).astype(np.float32)
    scale = np.float32(np.sqrt(output_dim))
    return (projection / scale).astype(np.float32, copy=False)


def standardize_fit(values):
    values = np.asarray(values, dtype=np.float64)
    mean = np.mean(values, axis=0)
    scale = np.std(values, axis=0)
    scale[scale < 1e-8] = 1.0
    return mean, scale


def fit_linear_readout(x_train, y_train, x_calibration, y_calibration):
    mean, scale = standardize_fit(x_train)
    train = (np.asarray(x_train, dtype=np.float64) - mean) / scale
    calibration = (np.asarray(x_calibration, dtype=np.float64) - mean) / scale
    train = np.column_stack([np.ones(len(train)), train])
    calibration = np.column_stack([np.ones(len(calibration)), calibration])
    gram = train.T @ train
    cross = train.T @ np.asarray(y_train, dtype=np.float64)
    best = None
    for ridge in RIDGE_LAMBDAS:
        penalty = np.eye(gram.shape[0]) * ridge
        penalty[0, 0] = 0.0
        coefficient = np.linalg.solve(gram + penalty, cross)
        prediction = calibration @ coefficient
        loss = float(np.mean((prediction - y_calibration) ** 2))
        candidate = {
            "ridge": float(ridge),
            "calibration_pose_mse": loss,
            "mean": mean,
            "scale": scale,
            "coefficient": coefficient,
        }
        if best is None or loss < best["calibration_pose_mse"]:
            best = candidate
    return best


def predict_linear_readout(probe, values):
    standardized = (
        np.asarray(values, dtype=np.float64) - probe["mean"]
    ) / probe["scale"]
    augmented = np.column_stack([np.ones(len(standardized)), standardized])
    return augmented @ probe["coefficient"]


def configure_repo():
    repo = CACHE_ROOT / "jepa-wms"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT],
        check=True,
    )
    resolved = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    # The public hub loader imports a planning entry point with extra evaluation
    # dependencies. Use its equivalent lightweight model constructor.
    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    # PushT and Wall do not use the DROID pose helper.
    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in simulator predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    config_paths = [
        repo
        / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/dino-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/jepa-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in config_paths:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo


def task_split_map():
    rng = np.random.default_rng(SEED + 17)
    order = rng.permutation(TASKS_PER_ENVIRONMENT).tolist()
    mapping = {}
    start = 0
    for name, count in zip(SPLIT_NAMES, TASK_SPLIT_COUNTS):
        for task_id in order[start : start + count]:
            mapping[int(task_id)] = name
        start += count
    return mapping


def pusht_tasks():
    # Frozen before any Stage 5 simulation. None appears in the Stage 3 family.
    values = [
        (220.0, 200.0, -0.90),
        (200.0, 272.0, 0.35),
        (224.0, 320.0, -0.30),
        (270.0, 198.0, 0.95),
        (292.0, 224.0, -0.45),
        (314.0, 270.0, 0.55),
        (292.0, 316.0, -1.00),
        (246.0, 310.0, 0.15),
        (230.0, 238.0, 1.25),
        (276.0, 244.0, -1.25),
        (314.0, 292.0, 0.85),
        (244.0, 278.0, -0.75),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "PushT",
            "task_id": index,
            "task_name": f"stage5_pusht_goal_{index:02d}",
            "goal": list(value),
            "split": splits[index],
        }
        for index, value in enumerate(values)
    ]


def wall_tasks():
    # Frozen before any Stage 5 simulation. None appears in the Stage 3 family.
    layouts = [
        (26.0, 20.0, 54.0, 44.0),
        (28.0, 34.0, 54.0, 16.0),
        (30.0, 44.0, 54.0, 32.0),
        (33.0, 16.0, 10.0, 48.0),
        (35.0, 28.0, 10.0, 20.0),
        (38.0, 42.0, 10.0, 36.0),
        (25.0, 38.0, 55.0, 22.0),
        (31.0, 24.0, 55.0, 50.0),
        (36.0, 46.0, 11.0, 16.0),
        (39.0, 20.0, 11.0, 40.0),
        (29.0, 30.0, 54.0, 50.0),
        (34.0, 38.0, 10.0, 24.0),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "Wall",
            "task_id": index,
            "task_name": f"stage5_wall_layout_goal_{index:02d}",
            "wall_x": wall_x,
            "door_y": door_y,
            "goal": [goal_x, goal_y],
            "split": splits[index],
        }
        for index, (wall_x, door_y, goal_x, goal_y) in enumerate(layouts)
    ]


TASKS = {"PushT": pusht_tasks(), "Wall": wall_tasks()}


def build_state_records(environment):
    tasks = TASKS[environment]
    per_task = NUM_STATES // TASKS_PER_ENVIRONMENT
    records = []
    state_id = 0
    for task in tasks:
        for within_task in range(per_task):
            evaluation_seed = EVALUATION_SEEDS[within_task % len(EVALUATION_SEEDS)]
            rng = np.random.default_rng(
                evaluation_seed * 100000
                + task["task_id"] * 1000
                + within_task
            )
            if environment == "PushT":
                goal_xy = np.asarray(task["goal"][:2], dtype=np.float64)
                for _ in range(100):
                    radial = rng.uniform(85.0, 120.0)
                    polar = rng.uniform(-np.pi, np.pi)
                    block = goal_xy + radial * np.array(
                        [np.cos(polar), np.sin(polar)]
                    )
                    direction = unit_vector(goal_xy - block)
                    agent_distance = rng.uniform(58.0, 80.0)
                    agent = block - agent_distance * direction
                    if (
                        np.all(block > 90.0)
                        and np.all(block < 422.0)
                        and np.all(agent > 35.0)
                        and np.all(agent < 477.0)
                    ):
                        break
                else:
                    raise RuntimeError("could not build bounded PushT state")
                state = np.array(
                    [
                        agent[0],
                        agent[1],
                        block[0],
                        block[1],
                        rng.uniform(-0.65, 0.65),
                        0.0,
                        0.0,
                    ],
                    dtype=np.float64,
                )
                stratum = "near" if agent_distance < 69.0 else "far"
            else:
                wall_x = float(task["wall_x"])
                goal_x = float(task["goal"][0])
                goal_on_right = goal_x > wall_x
                if goal_on_right:
                    x = rng.uniform(8.0, max(9.0, wall_x - 8.0))
                else:
                    x = rng.uniform(min(56.0, wall_x + 8.0), 57.0)
                y = rng.uniform(8.0, 57.0)
                state = np.array([x, y], dtype=np.float64)
                stratum = "left_to_right" if goal_on_right else "right_to_left"
            records.append(
                {
                    "environment": environment,
                    "state_id": state_id,
                    "task_id": task["task_id"],
                    "task_name": task["task_name"],
                    "split": task["split"],
                    "evaluation_seed": int(evaluation_seed),
                    "design_stratum": stratum,
                    "state": state,
                }
            )
            state_id += 1
    if len(records) != NUM_STATES:
        raise AssertionError("state-record count mismatch")
    return records


def pusht_candidate_library(state, task, primitive_steps):
    direction = unit_vector(
        np.asarray(task["goal"][:2]) - np.asarray(state)[2:4]
    )
    specifications = [("noop", 0.0, 0)]
    specifications.extend(
        (f"direct_{duration}", 0.0, duration)
        for duration in [8, 12, 16, 20, 24, 30]
    )
    for angle in [-20.0, 20.0]:
        for duration in [12, 18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-40.0, 40.0]:
        for duration in [18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-70.0, 70.0, 140.0, -140.0, 180.0]:
        specifications.append((f"angle_{angle:+.0f}_24", angle, 24))
    sequences = []
    for _, angle, duration in specifications:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        if duration:
            sequence[:duration] = (
                0.14 * rotate_vector(direction, angle)
            ).astype(np.float32)
        sequences.append(sequence)
    selected = np.asarray(
        [0, 2, 4, 6, 8, 11, 14, 16, 17, 18],
        dtype=np.int64,
    )
    return (
        np.stack(sequences)[selected],
        [specifications[index][0] for index in selected],
        selected,
    )


def nominal_waypoint_sequence(state, waypoints, primitive_steps, magnitude=0.75):
    position = np.asarray(state, dtype=np.float64).copy()
    sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
    waypoints = [np.asarray(point, dtype=np.float64) for point in waypoints]
    for step in range(primitive_steps):
        remaining = primitive_steps - step
        waypoint_index = min(
            len(waypoints) - 1,
            (step * len(waypoints)) // primitive_steps,
        )
        delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) < 0.5 and waypoint_index + 1 < len(waypoints):
            waypoint_index += 1
            delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) > 1e-8:
            action = magnitude * unit_vector(delta)
            sequence[step] = action.astype(np.float32)
            position = position + 2.0 * action
    return sequence


def wall_candidate_library(state, task, primitive_steps):
    state = np.asarray(state, dtype=np.float64)
    goal = np.asarray(task["goal"], dtype=np.float64)
    wall_x = float(task["wall_x"])
    door_y = float(task["door_y"])
    side = np.sign(goal[0] - state[0])
    door = np.array([wall_x + side * 1.0, door_y])
    directions = [
        ("noop", np.zeros((primitive_steps, 2), dtype=np.float32)),
        ("direct", nominal_waypoint_sequence(state, [goal], primitive_steps)),
        (
            "via_door",
            nominal_waypoint_sequence(state, [door, goal], primitive_steps),
        ),
        (
            "via_door_high",
            nominal_waypoint_sequence(
                state,
                [door + np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
        (
            "via_door_low",
            nominal_waypoint_sequence(
                state,
                [door - np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
    ]
    direct = unit_vector(goal - state)
    for angle in [-35.0, 35.0, -70.0, 70.0]:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        sequence[:] = (0.75 * rotate_vector(direct, angle)).astype(np.float32)
        directions.append((f"angle_{angle:+.0f}", sequence))
    reverse = np.zeros((primitive_steps, 2), dtype=np.float32)
    reverse[:] = (-0.55 * direct).astype(np.float32)
    directions.append(("reverse", reverse))
    if len(directions) != ACTIONS_PER_STATE:
        raise AssertionError("Wall candidate count mismatch")
    return (
        np.stack([item[1] for item in directions]),
        [item[0] for item in directions],
        np.arange(ACTIONS_PER_STATE, dtype=np.int64),
    )


def candidate_library(environment, state, task, primitive_steps):
    if environment == "PushT":
        return pusht_candidate_library(state, task, primitive_steps)
    return wall_candidate_library(state, task, primitive_steps)


def task_cost(environment, states, task):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle_error = np.arctan2(
            np.sin(states[..., 4] - goal[2]),
            np.cos(states[..., 4] - goal[2]),
        )
        pieces = np.concatenate(
            [
                (states[..., 2:4] - goal[:2]) / 512.0,
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    goal = np.asarray(task["goal"], dtype=np.float64)
    return np.linalg.norm((states[..., :2] - goal) / 65.0, axis=-1)


def pose_target(environment, states):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        angle = states[..., 4]
        return np.stack(
            [
                states[..., 2] / 512.0,
                states[..., 3] / 512.0,
                np.sin(angle),
                np.cos(angle),
            ],
            axis=-1,
        )
    return states[..., :2] / 65.0


def decoded_task_cost(environment, prediction, task):
    prediction = np.asarray(prediction, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_error = np.arctan2(
            np.sin(angle - goal[2]),
            np.cos(angle - goal[2]),
        )
        return np.linalg.norm(
            np.concatenate(
                [
                    prediction[..., :2] - goal[:2] / 512.0,
                    (angle_error / np.pi)[..., None],
                ],
                axis=-1,
            ),
            axis=-1,
        )
    return np.linalg.norm(
        prediction[..., :2] - np.asarray(task["goal"]) / 65.0,
        axis=-1,
    )


def physical_pose_error(environment, prediction, truth):
    prediction = np.asarray(prediction, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if environment == "PushT":
        angle_prediction = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_truth = np.arctan2(truth[..., 2], truth[..., 3])
        angle_error = np.arctan2(
            np.sin(angle_prediction - angle_truth),
            np.cos(angle_prediction - angle_truth),
        )
        pieces = np.concatenate(
            [
                prediction[..., :2] - truth[..., :2],
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    return np.linalg.norm(prediction[..., :2] - truth[..., :2], axis=-1)


def make_environment(repo, environment, task=None):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if environment == "PushT":
        from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv

        return PushTEnv(
            with_velocity=True,
            with_target=True,
            render_size=224,
            relative=True,
            action_scale=100,
        )

    from evals.simu_env_planning.envs.wall_gym_wrap import DEFAULT_CFG
    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall

    if task is None:
        task = TASKS["Wall"][0]
    env = DotWall(
        rng=np.random.default_rng(SEED),
        wall_config=deepcopy(DEFAULT_CFG),
        fix_wall=True,
        cross_wall=False,
        device="cpu",
    )
    env.wall_x = torch.tensor(float(task["wall_x"]))
    env.hole_y = torch.tensor(float(task["door_y"]))
    env.left_wall_x = env.wall_x - env.wall_config.wall_width // 2
    env.right_wall_x = env.wall_x + env.wall_config.wall_width // 2
    return env


def wall_visual(env):
    value = env.render().float()[None]
    resized = torch_functional.interpolate(
        value,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )[0]
    return (
        torch.clamp(torch.round(resized), 0, 255)
        .to(torch.uint8)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )


def reset_environment(repo, environment, task, state, seed):
    if environment == "PushT":
        env = make_environment(repo, environment)
        env.seed(seed)
        env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
        observation, restored = env.reset()
        payload = {
            "visual": np.asarray(observation["visual"]).copy(),
            "proprio": np.asarray(observation["proprio"]).copy(),
        }
        return env, payload, np.asarray(restored).copy()

    env = make_environment(repo, environment, task)
    env.seed(seed)
    env.reset_to_state = torch.as_tensor(
        np.asarray(state, dtype=np.float32)
    )
    observation, restored = env.reset()
    payload = {
        "visual": wall_visual(env),
        "proprio": np.asarray(
            observation["proprio"].detach().cpu(), dtype=np.float32
        ),
    }
    return env, payload, np.asarray(restored.detach().cpu(), dtype=np.float32)


def rollout_branch(repo, environment, task, state, actions, seed):
    env, initial, restored = reset_environment(
        repo, environment, task, state, seed
    )
    wanted = set(HORIZONS)
    observations = {}
    states = {}
    interactions = {}
    interaction_types = {}
    cumulative_interactions = 0
    cumulative_crossings = 0
    previous_state = np.asarray(restored, dtype=np.float64).copy()

    for step, action in enumerate(actions, start=1):
        if environment == "PushT":
            observation, _, _, info = env.step(action)
            current_state = np.asarray(info["state"]).copy()
            cumulative_interactions += int(info.get("n_contacts", 0))
            current_observation = {
                "visual": np.asarray(observation["visual"]).copy(),
                "proprio": np.asarray(observation["proprio"]).copy(),
            }
        else:
            observation, _, _, info = env.step(
                torch.as_tensor(action, dtype=torch.float32)
            )
            current_state = np.asarray(
                info["state"].detach().cpu(), dtype=np.float64
            )
            proposed = previous_state + 2.0 * np.asarray(action)
            if np.linalg.norm(current_state - proposed) > 1e-5:
                cumulative_interactions += 1
            wall_x = float(task["wall_x"])
            crossed = (
                (previous_state[0] - wall_x) * (current_state[0] - wall_x)
                < 0
            )
            cumulative_crossings += int(crossed)
            current_observation = {
                "visual": wall_visual(env),
                "proprio": np.asarray(
                    observation["proprio"].detach().cpu(),
                    dtype=np.float32,
                ),
            }
        previous_state = current_state.copy()
        if step % FRAMESKIP == 0:
            horizon = step // FRAMESKIP
            if horizon in wanted:
                observations[horizon] = current_observation
                states[horizon] = current_state
                interactions[horizon] = cumulative_interactions
                if environment == "PushT":
                    interaction_types[horizon] = (
                        "contact" if cumulative_interactions > 0 else "free"
                    )
                elif cumulative_interactions > 0:
                    interaction_types[horizon] = "collision"
                elif cumulative_crossings > 0:
                    interaction_types[horizon] = "door_cross"
                else:
                    interaction_types[horizon] = "free"
    if wanted != set(observations):
        raise RuntimeError(f"missing horizons: {wanted - set(observations)}")
    return (
        initial,
        restored,
        observations,
        states,
        interactions,
        interaction_types,
    )


def exact_restore_test(repo, environment, task, state, actions):
    endpoints = []
    images = []
    interactions = []
    for _ in range(3):
        initial, _, _, states, counts, kinds = rollout_branch(
            repo,
            environment,
            task,
            state,
            actions,
            SEED + 9000,
        )
        endpoints.append(states[max(HORIZONS)])
        images.append(initial["visual"])
        interactions.append(
            (counts[max(HORIZONS)], kinds[max(HORIZONS)])
        )
    result = {
        "environment": environment,
        "repeats": 3,
        "endpoint_bitwise_exact": all(
            np.array_equal(endpoints[0], item) for item in endpoints[1:]
        ),
        "initial_render_bitwise_exact": all(
            np.array_equal(images[0], item) for item in images[1:]
        ),
        "diagnostics_exact": all(
            interactions[0] == item for item in interactions[1:]
        ),
        "max_endpoint_abs_diff": float(
            max(
                np.max(np.abs(endpoints[0] - item))
                for item in endpoints[1:]
            )
        ),
    }
    if not all(
        result[key]
        for key in [
            "endpoint_bitwise_exact",
            "initial_render_bitwise_exact",
            "diagnostics_exact",
        ]
    ):
        raise AssertionError(f"exact restoration failed: {result}")
    return result


def goal_observation(repo, environment, task):
    if environment == "PushT":
        goal = task["goal"]
        state = np.array(
            [80.0, 450.0, goal[0], goal[1], goal[2], 0.0, 0.0]
        )
    else:
        state = np.asarray(task["goal"], dtype=np.float64)
    _, observation, _ = reset_environment(
        repo,
        environment,
        task,
        state,
        SEED + 11000 + task["task_id"],
    )
    return observation


In [ ]:
# Phase A — reconstruct the Stage 5 development interventions.
def generate_simulator_truth():
    repo = configure_repo()
    task_payload = []
    split_payload = {
        "protocol": (
            "prospective task-disjoint training/calibration/final-test; "
            "states are nested within exactly one task"
        ),
        "environments": {},
    }
    restore_payload = {}
    design_payload = {}

    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        truth_dir.mkdir(parents=True, exist_ok=True)
        records = build_state_records(environment)
        tasks_by_id = {
            item["task_id"]: item for item in TASKS[environment]
        }
        for task in TASKS[environment]:
            task_payload.append(task)

        split_payload["environments"][environment] = {
            name: {
                "task_ids": sorted(
                    task["task_id"]
                    for task in TASKS[environment]
                    if task["split"] == name
                ),
                "state_ids": sorted(
                    record["state_id"]
                    for record in records
                    if record["split"] == name
                ),
            }
            for name in SPLIT_NAMES
        }

        primitive_steps = max(HORIZONS) * FRAMESKIP
        first = records[0]
        first_task = tasks_by_id[first["task_id"]]
        first_actions, labels, selected = candidate_library(
            environment,
            first["state"],
            first_task,
            primitive_steps,
        )
        restore_payload[environment] = exact_restore_test(
            repo,
            environment,
            first_task,
            first["state"],
            first_actions[1],
        )

        state_matrix = []
        task_ids = []
        action_bank = []
        physical_costs = []
        interaction_bank = []
        interaction_type_bank = []
        for record in records:
            state_id = record["state_id"]
            state_path = truth_dir / f"state_{state_id:04d}.npz"
            if state_path.exists():
                log.info("%s simulator resume: keeping %s", environment, state_path.name)
                with np.load(state_path) as shard:
                    state_matrix.append(shard["initial_state"])
                    task_ids.append(int(shard["task_id"]))
                    action_bank.append(shard["selected_actions"])
                    physical_costs.append(shard["physical_cost"])
                    interaction_bank.append(shard["interactions"])
                    interaction_type_bank.append(shard["interaction_types"])
                continue

            task = tasks_by_id[record["task_id"]]
            actions, action_labels, selected_indices = candidate_library(
                environment,
                record["state"],
                task,
                primitive_steps,
            )
            if action_labels != labels:
                raise AssertionError("candidate labels changed across states")
            initials = []
            visuals = []
            proprios = []
            endpoints = []
            interactions = []
            interaction_types = []
            for branch in actions:
                (
                    initial,
                    _,
                    observations,
                    states,
                    counts,
                    kinds,
                ) = rollout_branch(
                    repo,
                    environment,
                    task,
                    record["state"],
                    branch,
                    record["evaluation_seed"] * 1000 + state_id,
                )
                initials.append(initial["visual"])
                visuals.append(
                    [observations[horizon]["visual"] for horizon in HORIZONS]
                )
                proprios.append(
                    [observations[horizon]["proprio"] for horizon in HORIZONS]
                )
                endpoints.append(
                    [states[horizon] for horizon in HORIZONS]
                )
                interactions.append(
                    [counts[horizon] for horizon in HORIZONS]
                )
                interaction_types.append(
                    [kinds[horizon] for horizon in HORIZONS]
                )
            if not all(
                np.array_equal(initials[0], item) for item in initials[1:]
            ):
                raise AssertionError(
                    f"branch initial render mismatch: {environment} state {state_id}"
                )
            endpoint_array = np.asarray(endpoints, dtype=np.float32)
            physical_cost = task_cost(
                environment, endpoint_array, task
            ).astype(np.float32)
            _, initial_observation, _ = reset_environment(
                repo,
                environment,
                task,
                record["state"],
                record["evaluation_seed"] * 1000 + state_id,
            )
            atomic_npz(
                state_path,
                initial_state=np.asarray(record["state"], dtype=np.float64),
                task_id=np.asarray(record["task_id"], dtype=np.int64),
                task_split=np.asarray(record["split"]),
                evaluation_seed=np.asarray(
                    record["evaluation_seed"], dtype=np.int64
                ),
                design_stratum=np.asarray(record["design_stratum"]),
                initial_visual=initials[0],
                initial_proprio=initial_observation["proprio"],
                selected_actions=actions,
                selected_library_indices=selected_indices,
                action_labels=np.asarray(action_labels),
                future_visual=np.asarray(visuals, dtype=np.uint8),
                future_proprio=np.asarray(proprios, dtype=np.float32),
                endpoint_states=endpoint_array,
                physical_cost=physical_cost,
                interactions=np.asarray(interactions, dtype=np.int32),
                interaction_types=np.asarray(interaction_types),
            )
            state_matrix.append(record["state"])
            task_ids.append(record["task_id"])
            action_bank.append(actions)
            physical_costs.append(physical_cost)
            interaction_bank.append(interactions)
            interaction_type_bank.append(interaction_types)
            write_json(
                OUT / f"{environment.lower()}_simulator_progress.json",
                {
                    "run_signature": RUN_SIGNATURE,
                    "environment": environment,
                    "completed_states": state_id + 1,
                    "total_states": NUM_STATES,
                    "last_file": state_path.name,
                },
            )
            log.info(
                "%s simulator state %d/%d",
                environment,
                state_id + 1,
                NUM_STATES,
            )

        physical_costs = np.asarray(physical_costs, dtype=np.float64)
        interactions = np.asarray(interaction_bank, dtype=np.int32)
        oracle = np.argmin(physical_costs, axis=1)
        spread = np.max(physical_costs, axis=1) - np.min(
            physical_costs, axis=1
        )
        no_op_regret = physical_costs[:, 0] - np.min(
            physical_costs, axis=1
        )
        left, right = pair_indices(ACTIONS_PER_STATE)
        pair_interactions = (
            (interactions[:, left, :] > 0).astype(int)
            + (interactions[:, right, :] > 0).astype(int)
        )
        environment_design = {
            "selection_protocol": (
                "fixed state/task-relative candidates; future simulator outcomes "
                "are never used for candidate selection"
            ),
            "candidate_labels": labels,
            "no_op_oracle_fraction_by_horizon": np.mean(
                oracle == 0, axis=0
            ).tolist(),
            "no_op_positive_regret_fraction_by_horizon": np.mean(
                no_op_regret > 1e-9, axis=0
            ).tolist(),
            "median_physical_cost_spread_by_horizon": np.median(
                spread, axis=0
            ).tolist(),
            "minimum_physical_cost_spread_by_horizon": np.min(
                spread, axis=0
            ).tolist(),
            "interaction_fraction_by_horizon": np.mean(
                interactions > 0, axis=(0, 1)
            ).tolist(),
            "pair_interaction_counts": {
                label: int(np.sum(pair_interactions == index))
                for index, label in enumerate(["neither", "one", "both"])
            },
        }
        environment_design["validity_thresholds"] = {
            "final_horizon_no_op_oracle_fraction_max": 0.25,
            "final_horizon_no_op_positive_regret_fraction_min": 0.75,
            "final_horizon_median_cost_spread_min": (
                0.08 if environment == "PushT" else 0.05
            ),
            "all_pair_interaction_strata_required": True,
        }
        environment_design["design_valid"] = bool(
            environment_design[
                "no_op_oracle_fraction_by_horizon"
            ][-1]
            < 0.25
            and environment_design[
                "no_op_positive_regret_fraction_by_horizon"
            ][-1]
            > 0.75
            and environment_design[
                "median_physical_cost_spread_by_horizon"
            ][-1]
            > (0.08 if environment == "PushT" else 0.05)
            and all(
                environment_design["pair_interaction_counts"][label] > 0
                for label in ["neither", "one", "both"]
            )
        )
        design_payload[environment] = environment_design
        atomic_npz(
            OUT / f"{environment.lower()}_design.npz",
            states=np.asarray(state_matrix),
            task_ids=np.asarray(task_ids),
            action_bank=np.asarray(action_bank, dtype=np.float32),
            physical_cost=physical_costs.astype(np.float32),
            interactions=interactions,
            interaction_types=np.asarray(interaction_type_bank),
            candidate_labels=np.asarray(labels),
        )

    write_json(OUT / "tasks.json", task_payload)
    write_json(OUT / "split_manifest.json", split_payload)
    write_json(OUT / "restore_test.json", restore_payload)
    write_json(OUT / "candidate_design_summary.json", design_payload)
    return repo


if not PIPELINE_FAILED:
    try:
        REPO = generate_simulator_truth()
    except Exception:
        record_failure("simulator_truth")


In [ ]:
# Phase B — frozen checkpoint evaluation and raw feature retention.
def evaluate_models():
    repo = configure_repo()
    checkpoint_records = []
    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        tasks_by_id = {
            item["task_id"]: item for item in TASKS[environment]
        }
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            model_dir = MODEL_ROOT / model_name
            model_dir.mkdir(parents=True, exist_ok=True)
            torch.cuda.reset_peak_memory_stats()
            gpu_report(f"{model_name}_before_load")
            model, preprocessor = torch.hub.load(
                str(repo),
                model_name,
                source="local",
                pretrained=True,
                device="cuda:0",
                trust_repo=True,
            )
            model.eval()
            gpu_report(f"{model_name}_after_load")

            goal_features = {}
            with torch.inference_mode():
                for task in TASKS[environment]:
                    observation = goal_observation(repo, environment, task)
                    encoded = model.encode(
                        to_model_observation(
                            observation["visual"],
                            observation["proprio"],
                        )
                    )
                    goal_features[task["task_id"]] = pool_visual(
                        encoded["visual"]
                    )[0, 0]

            horizon_index = torch.tensor(
                HORIZONS, dtype=torch.long, device="cuda"
            )
            for state_id in range(NUM_STATES):
                output_path = model_dir / f"state_{state_id:04d}.npz"
                if output_path.exists():
                    log.info(
                        "%s resume: keeping %s",
                        model_name,
                        output_path.name,
                    )
                    continue
                with np.load(
                    truth_dir / f"state_{state_id:04d}.npz"
                ) as truth:
                    task_id = int(truth["task_id"])
                    initial_visual = truth["initial_visual"]
                    initial_proprio = truth["initial_proprio"]
                    future_visual = truth["future_visual"]
                    future_proprio = truth["future_proprio"]
                    selected_actions = truth["selected_actions"]
                    physical_cost = truth["physical_cost"].astype(np.float64)

                chunks = torch.from_numpy(
                    selected_actions.reshape(
                        ACTIONS_PER_STATE,
                        max(HORIZONS),
                        FRAMESKIP,
                        2,
                    )
                ).float()
                normalized = preprocessor.normalize_actions(chunks)
                model_actions = (
                    normalized.reshape(
                        ACTIONS_PER_STATE,
                        max(HORIZONS),
                        -1,
                    )
                    .permute(1, 0, 2)
                    .contiguous()
                    .cuda()
                )
                with torch.inference_mode():
                    initial_encoded = model.encode(
                        to_model_observation(
                            initial_visual,
                            initial_proprio,
                        )
                    )
                    truth_encoded = model.encode(
                        to_model_observation(
                            future_visual,
                            future_proprio,
                        )
                    )
                    truth_features = pool_visual(truth_encoded["visual"])
                    predicted_encoded = model.unroll(
                        initial_encoded,
                        model_actions,
                    )
                    selected_prediction = predicted_encoded[
                        "visual"
                    ].index_select(0, horizon_index)
                    predicted_features = np.moveaxis(
                        pool_visual(selected_prediction),
                        0,
                        1,
                    )
                metrics = feature_metrics(
                    truth_features,
                    predicted_features,
                )
                goal_feature = goal_features[task_id]
                latent_true_cost = np.sqrt(
                    np.mean(
                        (
                            truth_features
                            - goal_feature[None, None, :]
                        )
                        ** 2,
                        axis=-1,
                    )
                )
                latent_predicted_cost = np.sqrt(
                    np.mean(
                        (
                            predicted_features
                            - goal_feature[None, None, :]
                        )
                        ** 2,
                        axis=-1,
                    )
                )
                atomic_npz(
                    output_path,
                    task_id=np.asarray(task_id),
                    predicted_features=predicted_features.astype(np.float16),
                    latent_true_cost=latent_true_cost.astype(np.float32),
                    latent_predicted_cost=latent_predicted_cost.astype(
                        np.float32
                    ),
                    physical_true_cost=physical_cost.astype(np.float32),
                    **{
                        key: np.asarray(value, dtype=np.float64)
                        for key, value in metrics.items()
                    },
                )
                write_json(
                    OUT / f"{model_name}_progress.json",
                    {
                        "run_signature": RUN_SIGNATURE,
                        "environment": environment,
                        "model": model_name,
                        "completed_states": state_id + 1,
                        "total_states": NUM_STATES,
                        "last_file": output_path.name,
                    },
                )
                log.info(
                    "%s state %d/%d",
                    model_name,
                    state_id + 1,
                    NUM_STATES,
                )
                if (state_id + 1) % 25 == 0:
                    gpu_report(f"{model_name}_state_{state_id:04d}")

            del model, preprocessor
            gc.collect()
            torch.cuda.empty_cache()
            gpu_report(f"{model_name}_released")

    hf_root = Path(os.environ["HF_HOME"]) / "hub"
    torch_root = Path(os.environ["TORCH_HOME"])
    for root in [hf_root, torch_root]:
        if root.exists():
            for path in root.rglob("*"):
                if (
                    path.is_file()
                    and path.stat().st_size > 20_000_000
                    and path.suffix
                    in {".tar", ".pth", ".pt", ".bin", ".safetensors"}
                ):
                    checkpoint_records.append(
                        {
                            "path": str(path),
                            "size_bytes": path.stat().st_size,
                            "sha256": sha256_file(path),
                        }
                    )
    write_json(
        OUT / "checkpoints_manifest.json",
        {
            "models": MODEL_NAME,
            "repository": "facebook/jepa-wms",
            "repository_commit": REPO_COMMIT,
            "training_seeds_per_public_configuration": 1,
            "training_seed_limitation": (
                "No additional public training-seed replicas are exposed by "
                "the checkpoint registry at the pinned commit."
            ),
            "cached_files": checkpoint_records,
        },
    )


if not PIPELINE_FAILED:
    try:
        evaluate_models()
    except Exception:
        record_failure("model_evaluation")


In [ ]:
# Phase C — structured action-effect adapter development.

ADAPTER_DIR = OUT / "trained_adapters"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

METHOD_WEIGHTS = {
    "endpoint_only": {
        "effect": 0.0,
        "independent": 0.0,
        "action": 0.0,
        "ranking": 0.0,
    },
    "action_decode_only": {
        "effect": 0.0,
        "independent": 0.0,
        "action": ACTION_DECODE_WEIGHT,
        "ranking": 0.0,
    },
    "ranking_only": {
        "effect": 0.0,
        "independent": 0.0,
        "action": 0.0,
        "ranking": RANK_WEIGHT,
    },
    "counterfactual_effect_only": {
        "effect": EFFECT_WEIGHT,
        "independent": 0.0,
        "action": 0.0,
        "ranking": 0.0,
    },
    "independent_action_effect": {
        "effect": 0.0,
        "independent": EFFECT_WEIGHT,
        "action": ACTION_DECODE_WEIGHT,
        "ranking": RANK_WEIGHT,
    },
    "counterfactual_action_effect": {
        "effect": EFFECT_WEIGHT,
        "independent": 0.0,
        "action": ACTION_DECODE_WEIGHT,
        "ranking": RANK_WEIGHT,
    },
}
if list(METHOD_WEIGHTS) != METHODS:
    raise AssertionError("METHODS and METHOD_WEIGHTS must have identical order")

MODEL_BY_ENVIRONMENT = {
    "PushT": ["dino_wm_pusht", "jepa_wm_pusht"],
    "Wall": ["dino_wm_wall", "jepa_wm_wall"],
}


def adapter_state_hash(model):
    digest = hashlib.sha256()
    for key, value in sorted(model.state_dict().items()):
        digest.update(key.encode())
        digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()


def load_stage6_truth(environment):
    truth_dir = TRUTH_ROOT / environment.lower()
    endpoints = []
    costs = []
    task_ids = []
    splits = []
    evaluation_seeds = []
    for state_id in range(NUM_STATES):
        with np.load(truth_dir / f"state_{state_id:04d}.npz") as shard:
            endpoints.append(shard["endpoint_states"])
            costs.append(shard["physical_cost"])
            task_ids.append(int(shard["task_id"]))
            splits.append(str(shard["task_split"]))
            evaluation_seeds.append(int(shard["evaluation_seed"]))
    with np.load(OUT / f"{environment.lower()}_design.npz") as design:
        action_bank = design["action_bank"].astype(np.float32)
    return {
        "pose": pose_target(
            environment, np.asarray(endpoints, dtype=np.float64)
        ).astype(np.float32),
        "physical_cost": np.asarray(costs, dtype=np.float32),
        "action_bank": action_bank,
        "task_id": np.asarray(task_ids, dtype=np.int64),
        "split": np.asarray(splits),
        "evaluation_seed": np.asarray(evaluation_seeds, dtype=np.int64),
    }


def load_raw_predicted_features(model_name):
    values = []
    model_dir = MODEL_ROOT / model_name
    for state_id in range(NUM_STATES):
        with np.load(model_dir / f"state_{state_id:04d}.npz") as shard:
            values.append(
                shard["predicted_features"].astype(np.float32)
            )
    values = np.asarray(values, dtype=np.float32)
    expected = (
        NUM_STATES,
        ACTIONS_PER_STATE,
        len(HORIZONS),
        values.shape[-1],
    )
    if values.shape != expected:
        raise AssertionError(
            f"unexpected predicted-feature shape {values.shape}; "
            f"expected {expected}"
        )
    return values


def build_action_descriptors(action_bank):
    action_bank = np.asarray(action_bank, dtype=np.float32)
    n_states = action_bank.shape[0]
    reshaped = action_bank.reshape(
        n_states,
        ACTIONS_PER_STATE,
        max(HORIZONS),
        FRAMESKIP,
        2,
    )
    descriptors = []
    for horizon in HORIZONS:
        prefix = reshaped[:, :, :horizon].reshape(
            n_states, ACTIONS_PER_STATE, horizon * FRAMESKIP, 2
        )
        displacement = np.sum(prefix, axis=2)
        mean_action = np.mean(prefix, axis=2)
        magnitude = np.linalg.norm(prefix, axis=-1)
        mean_magnitude = np.mean(magnitude, axis=2, keepdims=True)
        active_fraction = np.mean(
            magnitude > 1e-8, axis=2, keepdims=True
        ).astype(np.float32)
        descriptors.append(
            np.concatenate(
                [
                    displacement,
                    mean_action,
                    mean_magnitude,
                    active_fraction,
                ],
                axis=-1,
            )
        )
    return np.stack(descriptors, axis=2).astype(np.float32)


def normalized_task_goal(environment, task):
    if environment == "PushT":
        return np.asarray(
            [
                float(task["goal"][0]) / 512.0,
                float(task["goal"][1]) / 512.0,
                float(task["goal"][2]),
            ],
            dtype=np.float32,
        )
    return np.asarray(
        [
            float(task["goal"][0]) / 65.0,
            float(task["goal"][1]) / 65.0,
            0.0,
        ],
        dtype=np.float32,
    )


def stage6_state_units(environment, features, truth, split_name):
    state_indices = np.flatnonzero(truth["split"] == split_name)
    pose = truth["pose"][state_indices]
    cost = truth["physical_cost"][state_indices]
    descriptors = build_action_descriptors(
        truth["action_bank"][state_indices]
    )
    selected_features = features[state_indices]
    tasks_by_id = {
        int(item["task_id"]): item for item in TASKS[environment]
    }
    goals = np.stack(
        [
            normalized_task_goal(
                environment,
                tasks_by_id[int(truth["task_id"][state_id])],
            )
            for state_id in state_indices
        ]
    )
    n_states = len(state_indices)
    return {
        "features": np.transpose(
            selected_features, (0, 2, 1, 3)
        ).reshape(
            n_states * len(HORIZONS),
            ACTIONS_PER_STATE,
            selected_features.shape[-1],
        ).astype(np.float32),
        "pose": np.transpose(pose, (0, 2, 1, 3)).reshape(
            n_states * len(HORIZONS),
            ACTIONS_PER_STATE,
            pose.shape[-1],
        ).astype(np.float32),
        "physical_cost": np.transpose(cost, (0, 2, 1)).reshape(
            n_states * len(HORIZONS), ACTIONS_PER_STATE
        ).astype(np.float32),
        "action_descriptor": np.transpose(
            descriptors, (0, 2, 1, 3)
        ).reshape(
            n_states * len(HORIZONS),
            ACTIONS_PER_STATE,
            descriptors.shape[-1],
        ).astype(np.float32),
        "goal": np.repeat(goals, len(HORIZONS), axis=0).astype(
            np.float32
        ),
        "state_id": np.repeat(state_indices, len(HORIZONS)),
        "task_id": np.repeat(
            truth["task_id"][state_indices], len(HORIZONS)
        ),
        "evaluation_seed": np.repeat(
            truth["evaluation_seed"][state_indices], len(HORIZONS)
        ),
        "horizon": np.tile(
            np.asarray(HORIZONS, dtype=np.int64), n_states
        ),
        "horizon_index": np.tile(
            np.arange(len(HORIZONS), dtype=np.int64), n_states
        ),
    }


def fit_location_scale(values):
    values = np.asarray(values, dtype=np.float32)
    flat = values.reshape(-1, values.shape[-1])
    mean = np.mean(flat, axis=0, dtype=np.float64).astype(np.float32)
    scale = np.std(flat, axis=0, dtype=np.float64).astype(np.float32)
    scale[scale < 1e-6] = 1.0
    return mean, scale


class ActionEffectAdapter(torch.nn.Module):
    def __init__(
        self,
        input_dim,
        bottleneck_dim,
        hidden_dim,
        pose_dim,
        action_descriptor_dim,
        horizon_count,
    ):
        super().__init__()
        horizon_dim = 16
        self.feature_normalization = torch.nn.LayerNorm(
            input_dim, elementwise_affine=False
        )
        self.projector = torch.nn.Sequential(
            torch.nn.Linear(input_dim, bottleneck_dim),
            torch.nn.GELU(),
            torch.nn.LayerNorm(bottleneck_dim),
        )
        self.horizon_embedding = torch.nn.Embedding(
            horizon_count, horizon_dim
        )
        self.baseline_head = torch.nn.Sequential(
            torch.nn.Linear(
                bottleneck_dim + horizon_dim, hidden_dim
            ),
            torch.nn.GELU(),
            torch.nn.LayerNorm(hidden_dim),
            torch.nn.Linear(hidden_dim, pose_dim),
        )
        self.effect_head = torch.nn.Sequential(
            torch.nn.Linear(
                bottleneck_dim
                + action_descriptor_dim
                + horizon_dim,
                hidden_dim,
            ),
            torch.nn.GELU(),
            torch.nn.LayerNorm(hidden_dim),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, pose_dim),
        )
        self.action_decoder = torch.nn.Sequential(
            torch.nn.Linear(bottleneck_dim, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, action_descriptor_dim),
        )

    def forward(self, features, action_descriptor, horizon_index):
        projected = self.projector(
            self.feature_normalization(features)
        )
        context = torch.mean(projected, dim=1)
        centered = projected - context[:, None, :]
        horizon = self.horizon_embedding(horizon_index)
        baseline = self.baseline_head(
            torch.cat([context, horizon], dim=-1)
        )
        expanded_horizon = horizon[:, None, :].expand(
            -1, features.shape[1], -1
        )
        raw_effect = self.effect_head(
            torch.cat(
                [centered, action_descriptor, expanded_horizon],
                dim=-1,
            )
        )
        effect = raw_effect - raw_effect[:, :1]
        pose = baseline[:, None, :] + effect
        decoded_action = self.action_decoder(centered)
        return pose, decoded_action, effect


def differentiable_task_cost(environment, pose, goal):
    if environment == "PushT":
        angle = torch.atan2(pose[..., 2], pose[..., 3])
        angle_error = torch.atan2(
            torch.sin(angle - goal[:, None, 2]),
            torch.cos(angle - goal[:, None, 2]),
        )
        pieces = torch.cat(
            [
                pose[..., :2] - goal[:, None, :2],
                (angle_error / math.pi)[..., None],
            ],
            dim=-1,
        )
        return torch.linalg.vector_norm(pieces, dim=-1)
    return torch.linalg.vector_norm(
        pose[..., :2] - goal[:, None, :2], dim=-1
    )


def weighted_pairwise_ranking_loss(
    predicted_cost, true_cost, temperature
):
    left = torch.as_tensor(
        PAIR_LEFT, dtype=torch.long, device=predicted_cost.device
    )
    right = torch.as_tensor(
        PAIR_RIGHT, dtype=torch.long, device=predicted_cost.device
    )
    true_margin = true_cost[:, right] - true_cost[:, left]
    predicted_margin = (
        predicted_cost[:, right] - predicted_cost[:, left]
    )
    spread = (
        torch.max(true_cost, dim=1).values
        - torch.min(true_cost, dim=1).values
    ).clamp_min(1e-6)
    weights = torch.abs(true_margin) / spread[:, None]
    signs = torch.sign(true_margin)
    losses = torch.nn.functional.softplus(
        -signs * predicted_margin / temperature
    )
    valid = weights > 1e-7
    if not torch.any(valid):
        return predicted_cost.sum() * 0.0
    return torch.sum(losses[valid] * weights[valid]) / torch.sum(
        weights[valid]
    )


def stage6_objective_loss(
    method,
    environment,
    standardized_prediction,
    standardized_target,
    decoded_action,
    standardized_action,
    true_cost,
    goal,
    target_mean,
    target_scale,
):
    absolute = torch.nn.functional.smooth_l1_loss(
        standardized_prediction, standardized_target
    )
    predicted_effect = (
        standardized_prediction - standardized_prediction[:, :1]
    )
    true_effect = standardized_target - standardized_target[:, :1]
    counterfactual_effect = torch.nn.functional.smooth_l1_loss(
        predicted_effect, true_effect
    )

    partner_prediction = torch.roll(
        standardized_prediction[:, :1], shifts=1, dims=0
    )
    partner_target = torch.roll(
        standardized_target[:, :1], shifts=1, dims=0
    )
    independent_effect = torch.nn.functional.smooth_l1_loss(
        standardized_prediction - partner_prediction,
        standardized_target - partner_target,
    )
    action_decode = torch.nn.functional.smooth_l1_loss(
        decoded_action, standardized_action
    )
    physical_prediction = (
        standardized_prediction * target_scale + target_mean
    )
    predicted_cost = differentiable_task_cost(
        environment, physical_prediction, goal
    )
    ranking = weighted_pairwise_ranking_loss(
        predicted_cost, true_cost, RANK_TEMPERATURE
    )
    weights = METHOD_WEIGHTS[method]
    total = (
        absolute
        + weights["effect"] * counterfactual_effect
        + weights["independent"] * independent_effect
        + weights["action"] * action_decode
        + weights["ranking"] * ranking
    )
    return total, {
        "absolute_loss": absolute,
        "counterfactual_effect_loss": counterfactual_effect,
        "independent_effect_loss": independent_effect,
        "action_decode_loss": action_decode,
        "ranking_loss": ranking,
    }


def adapter_predictions(
    model,
    units,
    target_mean,
    target_scale,
    action_mean,
    action_scale,
):
    model.eval()
    values = []
    with torch.inference_mode():
        for start in range(0, len(units["features"]), 64):
            stop = start + 64
            features = torch.from_numpy(
                units["features"][start:stop]
            ).to(device="cuda", dtype=torch.float32)
            action = torch.from_numpy(
                units["action_descriptor"][start:stop]
            ).to(device="cuda", dtype=torch.float32)
            action = (
                action
                - torch.as_tensor(
                    action_mean, device="cuda", dtype=torch.float32
                )
            ) / torch.as_tensor(
                action_scale, device="cuda", dtype=torch.float32
            )
            horizon_index = torch.from_numpy(
                units["horizon_index"][start:stop]
            ).to(device="cuda", dtype=torch.long)
            standardized, _, _ = model(
                features, action, horizon_index
            )
            values.append(standardized.cpu().numpy())
    standardized = np.concatenate(values)
    return (
        standardized * target_scale[None, None, :]
        + target_mean[None, None, :]
    ).astype(np.float32)


def calibration_selection_score(
    environment,
    model,
    units,
    target_mean,
    target_scale,
    action_mean,
    action_scale,
):
    prediction = adapter_predictions(
        model,
        units,
        target_mean,
        target_scale,
        action_mean,
        action_scale,
    )
    tasks_by_id = {
        int(item["task_id"]): item for item in TASKS[environment]
    }
    regrets = []
    rankings = []
    pose_errors = []
    for index in range(len(prediction)):
        task = tasks_by_id[int(units["task_id"][index])]
        predicted_cost = decoded_task_cost(
            environment, prediction[index], task
        )
        ranking = ranking_metrics(
            units["physical_cost"][index], predicted_cost
        )
        regrets.append(ranking["normalized_regret"])
        if np.isfinite(ranking["weighted_pairwise_accuracy"]):
            rankings.append(
                ranking["weighted_pairwise_accuracy"]
            )
        pose_components = physical_pose_error(
            environment,
            prediction[index],
            units["pose"][index],
        )
        pose_errors.append(
            float(np.sqrt(np.mean(pose_components**2)))
        )
    if not rankings:
        return float("inf"), {}
    components = {
        "normalized_regret": float(np.mean(regrets)),
        "weighted_pairwise_accuracy": float(np.mean(rankings)),
        "pose_error": float(np.mean(pose_errors)),
    }
    score = (
        components["normalized_regret"]
        + (1.0 - components["weighted_pairwise_accuracy"])
        + 0.25 * components["pose_error"]
    )
    return float(score), components


def train_action_effect_adapter(
    environment,
    model_name,
    adapter_seed,
    method,
    train_units,
    calibration_units,
    target_mean,
    target_scale,
    action_mean,
    action_scale,
):
    group_seed = stable_seed(
        "stage6_adapter", environment, model_name, adapter_seed
    )
    random.seed(group_seed)
    np.random.seed(group_seed)
    torch.manual_seed(group_seed)
    torch.cuda.manual_seed_all(group_seed)
    model = ActionEffectAdapter(
        input_dim=train_units["features"].shape[-1],
        bottleneck_dim=ADAPTER_BOTTLENECK_DIM,
        hidden_dim=ADAPTER_HIDDEN_DIM,
        pose_dim=train_units["pose"].shape[-1],
        action_descriptor_dim=train_units[
            "action_descriptor"
        ].shape[-1],
        horizon_count=len(HORIZONS),
    ).cuda()
    initial_hash = adapter_state_hash(model)
    checkpoint = (
        ADAPTER_DIR
        / f"{model_name}_seed{adapter_seed}_{method}.pt"
    )
    if checkpoint.exists():
        cached = torch.load(
            checkpoint, map_location="cpu", weights_only=False
        )
        if (
            cached.get("run_signature") != RUN_SIGNATURE
            or cached.get("adapter_implementation_id")
            != ADAPTER_IMPLEMENTATION_ID
            or cached.get("initial_parameter_sha256") != initial_hash
            or cached.get("method") != method
        ):
            raise RuntimeError(
                f"incompatible cached adapter checkpoint: {checkpoint}"
            )
        model.load_state_dict(cached["state_dict"])
        model.cuda()
        cached_record = dict(cached["training_record"])
        cached_record["checkpoint_sha256"] = sha256_file(
            checkpoint
        )
        return (
            model,
            cached["history"],
            cached_record,
            cached["selection_records"],
        )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAINING_LR,
        weight_decay=TRAINING_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAINING_EPOCHS,
        eta_min=TRAINING_LR * 0.05,
    )
    target_mean_tensor = torch.as_tensor(
        target_mean, device="cuda", dtype=torch.float32
    )
    target_scale_tensor = torch.as_tensor(
        target_scale, device="cuda", dtype=torch.float32
    )
    action_mean_tensor = torch.as_tensor(
        action_mean, device="cuda", dtype=torch.float32
    )
    action_scale_tensor = torch.as_tensor(
        action_scale, device="cuda", dtype=torch.float32
    )

    history = []
    selection_records = []
    candidate_states = {}
    updates = 0
    for epoch in range(1, TRAINING_EPOCHS + 1):
        model.train()
        generator = np.random.default_rng(
            stable_seed("stage6_batches", group_seed, epoch)
        )
        order = generator.permutation(len(train_units["features"]))
        component_values = {
            "loss": [],
            "absolute_loss": [],
            "counterfactual_effect_loss": [],
            "independent_effect_loss": [],
            "action_decode_loss": [],
            "ranking_loss": [],
        }
        for start in range(0, len(order), TRAINING_BATCH_UNITS):
            indices = order[start : start + TRAINING_BATCH_UNITS]
            features = torch.from_numpy(
                train_units["features"][indices]
            ).to(device="cuda", dtype=torch.float32)
            target = torch.from_numpy(
                train_units["pose"][indices]
            ).to(device="cuda", dtype=torch.float32)
            standardized_target = (
                target - target_mean_tensor
            ) / target_scale_tensor
            action = torch.from_numpy(
                train_units["action_descriptor"][indices]
            ).to(device="cuda", dtype=torch.float32)
            standardized_action = (
                action - action_mean_tensor
            ) / action_scale_tensor
            horizon_index = torch.from_numpy(
                train_units["horizon_index"][indices]
            ).to(device="cuda", dtype=torch.long)
            true_cost = torch.from_numpy(
                train_units["physical_cost"][indices]
            ).to(device="cuda", dtype=torch.float32)
            goal = torch.from_numpy(
                train_units["goal"][indices]
            ).to(device="cuda", dtype=torch.float32)

            optimizer.zero_grad(set_to_none=True)
            prediction, decoded_action, _ = model(
                features, standardized_action, horizon_index
            )
            loss, components = stage6_objective_loss(
                method,
                environment,
                prediction,
                standardized_target,
                decoded_action,
                standardized_action,
                true_cost,
                goal,
                target_mean_tensor,
                target_scale_tensor,
            )
            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"non-finite Stage 6 loss: {environment} "
                    f"{model_name} {adapter_seed} {method} epoch {epoch}"
                )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            updates += 1
            component_values["loss"].append(
                float(loss.detach().cpu())
            )
            for key, value in components.items():
                component_values[key].append(
                    float(value.detach().cpu())
                )
        scheduler.step()
        history.append(
            {
                "environment": environment,
                "model": model_name,
                "adapter_seed": int(adapter_seed),
                "method": method,
                "epoch": int(epoch),
                **{
                    key: float(np.mean(values))
                    for key, values in component_values.items()
                },
                "learning_rate": float(
                    scheduler.get_last_lr()[0]
                ),
                "completed_updates": int(updates),
            }
        )
        if epoch in SELECTION_EPOCHS:
            score, components = calibration_selection_score(
                environment,
                model,
                calibration_units,
                target_mean,
                target_scale,
                action_mean,
                action_scale,
            )
            selection_records.append(
                {
                    "environment": environment,
                    "model": model_name,
                    "adapter_seed": int(adapter_seed),
                    "method": method,
                    "epoch": int(epoch),
                    "selection_score": float(score),
                    **components,
                }
            )
            candidate_states[epoch] = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
    if set(candidate_states) != set(SELECTION_EPOCHS):
        raise AssertionError("not all calibration checkpoints were evaluated")
    selected = min(
        selection_records,
        key=lambda row: (row["selection_score"], row["epoch"]),
    )
    selected_epoch = int(selected["epoch"])
    model.load_state_dict(candidate_states[selected_epoch])
    model.cuda()
    for row in history:
        row["selected_epoch"] = selected_epoch
    for row in selection_records:
        row["selected"] = bool(row["epoch"] == selected_epoch)
    training_record = {
        "environment": environment,
        "model": model_name,
        "adapter_seed": int(adapter_seed),
        "method": method,
        "initial_parameter_sha256": initial_hash,
        "completed_updates": int(updates),
        "training_units": int(len(train_units["features"])),
        "maximum_epochs": int(TRAINING_EPOCHS),
        "selected_epoch": selected_epoch,
        "selected_calibration_score": float(
            selected["selection_score"]
        ),
        "checkpoint": checkpoint.relative_to(OUT).as_posix(),
    }
    payload = {
        "run_signature": RUN_SIGNATURE,
        "adapter_implementation_id": ADAPTER_IMPLEMENTATION_ID,
        "method": method,
        "state_dict": {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
        "target_mean": target_mean,
        "target_scale": target_scale,
        "action_mean": action_mean,
        "action_scale": action_scale,
        "initial_parameter_sha256": initial_hash,
        "history": history,
        "selection_records": selection_records,
        "training_record": training_record,
    }
    temporary = checkpoint.with_suffix(".tmp.pt")
    torch.save(payload, temporary)
    temporary.replace(checkpoint)
    training_record["checkpoint_sha256"] = sha256_file(checkpoint)
    return model, history, training_record, selection_records


def evaluate_stage6_adapter(
    environment,
    model_name,
    adapter_seed,
    method,
    split_label,
    model,
    units,
    target_mean,
    target_scale,
    action_mean,
    action_scale,
):
    predictions = adapter_predictions(
        model,
        units,
        target_mean,
        target_scale,
        action_mean,
        action_scale,
    )
    tasks_by_id = {
        int(item["task_id"]): item for item in TASKS[environment]
    }
    unit_rows = []
    action_rows = []
    for index in range(len(predictions)):
        task = tasks_by_id[int(units["task_id"][index])]
        predicted_pose = predictions[index].astype(np.float64)
        true_pose = units["pose"][index].astype(np.float64)
        true_cost = units["physical_cost"][index].astype(np.float64)
        predicted_cost = decoded_task_cost(
            environment, predicted_pose, task
        )
        ranking = ranking_metrics(true_cost, predicted_cost)
        pose_components = physical_pose_error(
            environment, predicted_pose, true_pose
        )
        pose_error = float(
            np.sqrt(np.mean(pose_components**2))
        )
        cost_rmse = float(
            np.sqrt(np.mean((predicted_cost - true_cost) ** 2))
        )
        common = {
            "environment": environment,
            "state_id": int(units["state_id"][index]),
            "task_id": int(units["task_id"][index]),
            "split": split_label,
            "evaluation_seed": int(
                units["evaluation_seed"][index]
            ),
            "model": model_name,
            "model_family": (
                "DINO-WM"
                if model_name.startswith("dino")
                else "JEPA-WM"
            ),
            "adapter_seed": int(adapter_seed),
            "method": method,
            "horizon": int(units["horizon"][index]),
        }
        unit_rows.append(
            {
                **common,
                "pose_error": pose_error,
                "physical_cost_rmse": cost_rmse,
                "normalized_regret": ranking[
                    "normalized_regret"
                ],
                "weighted_pairwise_accuracy": ranking[
                    "weighted_pairwise_accuracy"
                ],
                "ranking_defined": bool(
                    np.isfinite(
                        ranking["weighted_pairwise_accuracy"]
                    )
                ),
                "top1_correct": ranking["top1_correct"],
                "normalized_margin_rmse": ranking[
                    "normalized_margin_rmse"
                ],
                "selected_action": ranking["selected_action"],
                "oracle_action": ranking["oracle_action"],
            }
        )
        for action in range(ACTIONS_PER_STATE):
            true_values = np.full(4, np.nan)
            predicted_values = np.full(4, np.nan)
            true_values[: true_pose.shape[-1]] = true_pose[action]
            predicted_values[: predicted_pose.shape[-1]] = (
                predicted_pose[action]
            )
            action_rows.append(
                {
                    **common,
                    "action": int(action),
                    "true_cost": float(true_cost[action]),
                    "predicted_cost": float(predicted_cost[action]),
                    **{
                        f"true_pose_{dim}": float(true_values[dim])
                        for dim in range(4)
                    },
                    **{
                        f"predicted_pose_{dim}": float(
                            predicted_values[dim]
                        )
                        for dim in range(4)
                    },
                }
            )
    return unit_rows, action_rows


all_training_history = []
all_selection_rows = []
all_training_records = []
all_unit_rows = []
all_action_rows = []
adapter_design_records = []

if not PIPELINE_FAILED:
    try:
        for environment in ENVIRONMENT:
            truth = load_stage6_truth(environment)
            for model_name in MODEL_BY_ENVIRONMENT[environment]:
                raw = load_raw_predicted_features(model_name)
                for adapter_seed in PROBE_SEEDS:
                    split_units = {
                        "probe_train": stage6_state_units(
                            environment, raw, truth, "probe_train"
                        ),
                        "probe_calibration": stage6_state_units(
                            environment,
                            raw,
                            truth,
                            "probe_calibration",
                        ),
                        "development_holdout": stage6_state_units(
                            environment, raw, truth, "final_test"
                        ),
                    }
                    target_mean, target_scale = fit_location_scale(
                        split_units["probe_train"]["pose"]
                    )
                    action_mean, action_scale = fit_location_scale(
                        split_units["probe_train"][
                            "action_descriptor"
                        ]
                    )
                    adapter_design_records.append(
                        {
                            "environment": environment,
                            "model": model_name,
                            "adapter_seed": int(adapter_seed),
                            "raw_feature_dim": int(raw.shape[-1]),
                            "learned_bottleneck_dim": int(
                                ADAPTER_BOTTLENECK_DIM
                            ),
                            "action_descriptor_dim": int(
                                split_units["probe_train"][
                                    "action_descriptor"
                                ].shape[-1]
                            ),
                            "training_state_clusters": int(
                                len(
                                    np.unique(
                                        split_units["probe_train"][
                                            "state_id"
                                        ]
                                    )
                                )
                            ),
                            "calibration_state_clusters": int(
                                len(
                                    np.unique(
                                        split_units[
                                            "probe_calibration"
                                        ]["state_id"]
                                    )
                                )
                            ),
                            "development_state_clusters": int(
                                len(
                                    np.unique(
                                        split_units[
                                            "development_holdout"
                                        ]["state_id"]
                                    )
                                )
                            ),
                        }
                    )
                    for method in METHODS:
                        (
                            model,
                            history,
                            training_record,
                            selection_records,
                        ) = train_action_effect_adapter(
                            environment,
                            model_name,
                            adapter_seed,
                            method,
                            split_units["probe_train"],
                            split_units["probe_calibration"],
                            target_mean,
                            target_scale,
                            action_mean,
                            action_scale,
                        )
                        all_training_history.extend(history)
                        all_training_records.append(training_record)
                        all_selection_rows.extend(selection_records)
                        for split_label in [
                            "probe_calibration",
                            "development_holdout",
                        ]:
                            unit_rows, action_rows = (
                                evaluate_stage6_adapter(
                                    environment,
                                    model_name,
                                    adapter_seed,
                                    method,
                                    split_label,
                                    model,
                                    split_units[split_label],
                                    target_mean,
                                    target_scale,
                                    action_mean,
                                    action_scale,
                                )
                            )
                            all_unit_rows.extend(unit_rows)
                            all_action_rows.extend(action_rows)
                        del model
                        gc.collect()
                        torch.cuda.empty_cache()
                    gpu_report(
                        f"{model_name}_seed{adapter_seed}_"
                        "stage6_training_complete"
                    )
                del raw
                gc.collect()
        write_csv(
            OUT / "adapter_training_history.csv",
            all_training_history,
        )
        write_csv(
            OUT / "checkpoint_selection.csv",
            all_selection_rows,
        )
        write_csv(OUT / "unit_metrics.csv", all_unit_rows)
        write_csv(
            OUT / "action_predictions.csv", all_action_rows
        )
        write_json(
            OUT / "adapter_training_manifest.json",
            {
                "evidence_status": EVIDENCE_STATUS,
                "world_models_frozen": True,
                "learned_projection": True,
                "set_aware_architecture": True,
                "training_partition": "probe_train",
                "calibration_used_for_checkpoint_selection": True,
                "development_holdout_used_for_selection": False,
                "methods": METHODS,
                "method_weights": METHOD_WEIGHTS,
                "selection_epochs": SELECTION_EPOCHS,
                "adapter_design_records": adapter_design_records,
                "training_records": all_training_records,
            },
        )
        (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")
    except Exception:
        record_failure("stage6_action_effect_adapter_training")


In [ ]:
# Phase D — exploratory development-holdout analysis.

LOWER_IS_BETTER = {
    "pose_error",
    "physical_cost_rmse",
    "normalized_regret",
    "normalized_margin_rmse",
}
DEVELOPMENT_SPLIT = "development_holdout"
PROPOSED_METHOD = "counterfactual_action_effect"
PRIMARY_BASELINE = "endpoint_only"
SPECIFICITY_BASELINE = "independent_action_effect"


def finite_mean(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.mean(values)) if len(values) else float("nan")


def percentile_interval(draws):
    draws = np.asarray(draws, dtype=np.float64)
    draws = draws[np.isfinite(draws)]
    if not len(draws):
        return float("nan"), float("nan")
    return (
        float(np.quantile(draws, 0.025)),
        float(np.quantile(draws, 0.975)),
    )


def summarize_stage6(rows):
    metrics = [
        "pose_error",
        "physical_cost_rmse",
        "normalized_regret",
        "weighted_pairwise_accuracy",
        "top1_correct",
        "normalized_margin_rmse",
    ]
    output = []
    for environment in ENVIRONMENT:
        for split_name in [
            "probe_calibration",
            DEVELOPMENT_SPLIT,
        ]:
            for method in METHODS:
                selected = [
                    row
                    for row in rows
                    if row["environment"] == environment
                    and row["split"] == split_name
                    and row["method"] == method
                ]
                if not selected:
                    continue
                output.append(
                    {
                        "environment": environment,
                        "split": split_name,
                        "method": method,
                        "n_rows": int(len(selected)),
                        "n_state_clusters": int(
                            len(
                                {
                                    int(row["state_id"])
                                    for row in selected
                                }
                            )
                        ),
                        **{
                            metric: finite_mean(
                                [row[metric] for row in selected]
                            )
                            for metric in metrics
                        },
                    }
                )
    return output


def state_metric_map(rows, environment, method, metric):
    grouped = {}
    for row in rows:
        if (
            row["environment"] != environment
            or row["split"] != DEVELOPMENT_SPLIT
            or row["method"] != method
        ):
            continue
        value = float(row[metric])
        if np.isfinite(value):
            grouped.setdefault(int(row["state_id"]), []).append(value)
    return {
        state_id: float(np.mean(values))
        for state_id, values in grouped.items()
    }


def clustered_method_contrast(
    rows,
    environment,
    comparator,
    proposed,
    metric,
    repetitions,
    seed,
):
    comparator_values = state_metric_map(
        rows, environment, comparator, metric
    )
    proposed_values = state_metric_map(
        rows, environment, proposed, metric
    )
    common = sorted(
        set(comparator_values) & set(proposed_values)
    )
    if not common:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": int(repetitions),
        }
    comparator_array = np.asarray(
        [comparator_values[state] for state in common],
        dtype=np.float64,
    )
    proposed_array = np.asarray(
        [proposed_values[state] for state in common],
        dtype=np.float64,
    )
    if metric in LOWER_IS_BETTER:
        difference = comparator_array - proposed_array
    else:
        difference = proposed_array - comparator_array
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions, dtype=np.float64)
    for index in range(repetitions):
        sampled = rng.integers(0, len(common), len(common))
        draws[index] = np.mean(difference[sampled])
    low, high = percentile_interval(draws)
    return {
        "estimate": float(np.mean(difference)),
        "low": low,
        "high": high,
        "n_clusters": int(len(common)),
        "n_bootstrap": int(repetitions),
    }


def clustered_pose_ratio(
    rows,
    environment,
    numerator,
    denominator,
    repetitions,
    seed,
):
    numerator_values = state_metric_map(
        rows, environment, numerator, "pose_error"
    )
    denominator_values = state_metric_map(
        rows, environment, denominator, "pose_error"
    )
    common = sorted(
        set(numerator_values) & set(denominator_values)
    )
    if not common:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": int(repetitions),
        }
    numerator_array = np.asarray(
        [numerator_values[state] for state in common]
    )
    denominator_array = np.asarray(
        [denominator_values[state] for state in common]
    )
    estimate = float(
        np.mean(numerator_array) / np.mean(denominator_array)
    )
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions, dtype=np.float64)
    for index in range(repetitions):
        sampled = rng.integers(0, len(common), len(common))
        draws[index] = np.mean(
            numerator_array[sampled]
        ) / np.mean(denominator_array[sampled])
    low, high = percentile_interval(draws)
    return {
        "estimate": estimate,
        "low": low,
        "high": high,
        "n_clusters": int(len(common)),
        "n_bootstrap": int(repetitions),
    }


def training_group_integrity(records):
    grouped = {}
    for row in records:
        key = (
            row["environment"],
            row["model"],
            int(row["adapter_seed"]),
        )
        grouped.setdefault(key, []).append(row)
    checks = []
    for (environment, model_name, adapter_seed), values in sorted(
        grouped.items()
    ):
        checks.append(
            {
                "environment": environment,
                "model": model_name,
                "adapter_seed": adapter_seed,
                "methods": sorted(row["method"] for row in values),
                "initial_hash_count": len(
                    {
                        row["initial_parameter_sha256"]
                        for row in values
                    }
                ),
                "update_count_count": len(
                    {int(row["completed_updates"]) for row in values}
                ),
                "training_unit_count_count": len(
                    {int(row["training_units"]) for row in values}
                ),
                "maximum_epoch_count": len(
                    {int(row["maximum_epochs"]) for row in values}
                ),
            }
        )
    for row in checks:
        row["pass"] = bool(
            row["methods"] == sorted(METHODS)
            and row["initial_hash_count"] == 1
            and row["update_count_count"] == 1
            and row["training_unit_count_count"] == 1
            and row["maximum_epoch_count"] == 1
        )
    return checks


stage6_summary_rows = []
stage6_contrast_rows = []
stage6_pose_ratio_rows = []
stage6_decision = {}

if not PIPELINE_FAILED:
    try:
        stage6_summary_rows = summarize_stage6(all_unit_rows)
        comparison_metrics = [
            "normalized_regret",
            "weighted_pairwise_accuracy",
            "top1_correct",
            "normalized_margin_rmse",
        ]
        for environment in ENVIRONMENT:
            for comparator in [
                method
                for method in METHODS
                if method != PROPOSED_METHOD
            ]:
                for metric in comparison_metrics:
                    result = clustered_method_contrast(
                        all_unit_rows,
                        environment,
                        comparator,
                        PROPOSED_METHOD,
                        metric,
                        BOOTSTRAP_REPS,
                        stable_seed(
                            "stage6_contrast",
                            environment,
                            comparator,
                            metric,
                        ),
                    )
                    stage6_contrast_rows.append(
                        {
                            "environment": environment,
                            "comparator": comparator,
                            "proposed": PROPOSED_METHOD,
                            "metric": metric,
                            "positive_means_proposed_better": True,
                            **result,
                        }
                    )
            ratio = clustered_pose_ratio(
                all_unit_rows,
                environment,
                PROPOSED_METHOD,
                PRIMARY_BASELINE,
                BOOTSTRAP_REPS,
                stable_seed("stage6_pose_ratio", environment),
            )
            stage6_pose_ratio_rows.append(
                {
                    "environment": environment,
                    "numerator": PROPOSED_METHOD,
                    "denominator": PRIMARY_BASELINE,
                    "metric": "pose_error",
                    "margin": float(POSE_ERROR_RATIO_MARGIN),
                    **ratio,
                    "pass": bool(
                        np.isfinite(ratio["high"])
                        and ratio["high"]
                        <= POSE_ERROR_RATIO_MARGIN
                    ),
                }
            )

        contrast_lookup = {
            (
                row["environment"],
                row["comparator"],
                row["metric"],
            ): row
            for row in stage6_contrast_rows
        }
        ratio_lookup = {
            row["environment"]: row
            for row in stage6_pose_ratio_rows
        }
        environment_gates = {}
        for environment in ENVIRONMENT:
            endpoint_regret = contrast_lookup[
                (
                    environment,
                    PRIMARY_BASELINE,
                    "normalized_regret",
                )
            ]
            endpoint_ranking = contrast_lookup[
                (
                    environment,
                    PRIMARY_BASELINE,
                    "weighted_pairwise_accuracy",
                )
            ]
            independent_regret = contrast_lookup[
                (
                    environment,
                    SPECIFICITY_BASELINE,
                    "normalized_regret",
                )
            ]
            independent_ranking = contrast_lookup[
                (
                    environment,
                    SPECIFICITY_BASELINE,
                    "weighted_pairwise_accuracy",
                )
            ]
            endpoint_planning = bool(
                endpoint_regret["low"] > 0
                and endpoint_ranking["low"] > 0
            )
            pose_noninferiority = bool(
                ratio_lookup[environment]["pass"]
            )
            specificity = bool(
                independent_regret["low"] > 0
                and independent_ranking["low"] > 0
            )
            environment_gates[environment] = {
                "endpoint_planning_pass": endpoint_planning,
                "pose_error_noninferiority_pass": pose_noninferiority,
                "complete_endpoint_gate_pass": bool(
                    endpoint_planning and pose_noninferiority
                ),
                "independent_control_specificity_pass": specificity,
            }

        design = json.loads(
            (OUT / "candidate_design_summary.json").read_text()
        )
        restore = json.loads(
            (OUT / "restore_test.json").read_text()
        )
        group_checks = training_group_integrity(
            all_training_records
        )
        expected_groups = (
            len(ENVIRONMENT) * 2 * len(PROBE_SEEDS)
        )
        selected_rows = [
            row for row in all_selection_rows if row["selected"]
        ]
        selected_epoch_valid = bool(
            len(selected_rows)
            == expected_groups * len(METHODS)
            and all(
                int(row["epoch"]) in SELECTION_EPOCHS
                for row in selected_rows
            )
        )
        final_cluster_counts = {
            environment: len(
                {
                    int(row["state_id"])
                    for row in all_unit_rows
                    if row["environment"] == environment
                    and row["split"] == DEVELOPMENT_SPLIT
                }
            )
            for environment in ENVIRONMENT
        }
        finite_ranking_clusters = {
            environment: len(
                {
                    int(row["state_id"])
                    for row in all_unit_rows
                    if row["environment"] == environment
                    and row["split"] == DEVELOPMENT_SPLIT
                    and row["method"] == PROPOSED_METHOD
                    and bool(row["ranking_defined"])
                }
            )
            for environment in ENVIRONMENT
        }
        finite_required = all(
            np.isfinite(
                [
                    row["pose_error"],
                    row["physical_cost_rmse"],
                    row["normalized_regret"],
                    row["top1_correct"],
                ]
            ).all()
            for row in all_unit_rows
        )
        task_sets = {
            environment: {
                split_name: {
                    int(task["task_id"])
                    for task in TASKS[environment]
                    if task["split"] == split_name
                }
                for split_name in [
                    "probe_train",
                    "probe_calibration",
                    "final_test",
                ]
            }
            for environment in ENVIRONMENT
        }
        no_training_or_calibration_overlap = all(
            not (
                task_sets[environment]["probe_train"]
                & task_sets[environment]["probe_calibration"]
            )
            and not (
                task_sets[environment]["probe_train"]
                & task_sets[environment]["final_test"]
            )
            and not (
                task_sets[environment]["probe_calibration"]
                & task_sets[environment]["final_test"]
            )
            for environment in ENVIRONMENT
        )
        integrity = {
            "evidence_status": EVIDENCE_STATUS,
            "candidate_design_pass": bool(
                all(
                    design[environment]["design_valid"]
                    for environment in ENVIRONMENT
                )
            ),
            "exact_restore_pass": bool(
                all(
                    restore[environment]["endpoint_bitwise_exact"]
                    and restore[environment][
                        "initial_render_bitwise_exact"
                    ]
                    and restore[environment]["diagnostics_exact"]
                    for environment in ENVIRONMENT
                )
            ),
            "training_groups_equal": bool(
                len(group_checks) == expected_groups
                and all(row["pass"] for row in group_checks)
            ),
            "training_group_checks": group_checks,
            "selected_epoch_valid": selected_epoch_valid,
            "required_metrics_finite": bool(finite_required),
            "development_state_clusters": final_cluster_counts,
            "finite_ranking_state_clusters": finite_ranking_clusters,
            "all_development_clusters_present": bool(
                all(
                    final_cluster_counts[environment]
                    == TASK_SPLIT_COUNTS[-1]
                    * (NUM_STATES // TASKS_PER_ENVIRONMENT)
                    for environment in ENVIRONMENT
                )
            ),
            "enough_finite_ranking_clusters": bool(
                all(
                    finite_ranking_clusters[environment] >= 20
                    for environment in ENVIRONMENT
                )
            ),
            "task_partitions_disjoint": bool(
                no_training_or_calibration_overlap
            ),
            "calibration_used_for_checkpoint_selection": True,
            "development_holdout_used_for_selection": False,
            "stage5_final_tasks_reused_and_not_confirmatory": True,
        }
        integrity["pass"] = bool(
            integrity["candidate_design_pass"]
            and integrity["exact_restore_pass"]
            and integrity["training_groups_equal"]
            and integrity["selected_epoch_valid"]
            and integrity["required_metrics_finite"]
            and integrity["all_development_clusters_present"]
            and integrity["enough_finite_ranking_clusters"]
            and integrity["task_partitions_disjoint"]
        )

        complete_count = sum(
            environment_gates[environment][
                "complete_endpoint_gate_pass"
            ]
            for environment in ENVIRONMENT
        )
        specificity_all = all(
            environment_gates[environment][
                "independent_control_specificity_pass"
            ]
            for environment in ENVIRONMENT
        )
        if not integrity["pass"]:
            status = "INCONCLUSIVE"
        elif complete_count == len(ENVIRONMENT) and specificity_all:
            status = "DEVELOPMENT_CANDIDATE_READY"
        elif complete_count == len(ENVIRONMENT):
            status = "PROMISING_BUT_NOT_SPECIFIC"
        elif complete_count == 1:
            status = "MIXED_DEVELOPMENT_SIGNAL"
        else:
            status = "NO_DEVELOPMENT_GAIN"
        stage6_decision = {
            "status": status,
            "evidence_status": EVIDENCE_STATUS,
            "proposed_method": PROPOSED_METHOD,
            "primary_baseline": PRIMARY_BASELINE,
            "specificity_baseline": SPECIFICITY_BASELINE,
            "pose_error_ratio_margin": float(
                POSE_ERROR_RATIO_MARGIN
            ),
            "environment_gates": environment_gates,
            "contrasts": {
                environment: {
                    comparator: {
                        metric: contrast_lookup[
                            (environment, comparator, metric)
                        ]
                        for metric in comparison_metrics
                    }
                    for comparator in [
                        method
                        for method in METHODS
                        if method != PROPOSED_METHOD
                    ]
                }
                for environment in ENVIRONMENT
            },
            "pose_error_noninferiority": ratio_lookup,
            "integrity": integrity,
            "claim_boundary": (
                "Stage 6 reuses already-inspected Stage 5 tasks for "
                "exploratory development. A positive result nominates a "
                "candidate for a new prospective Stage 6B task family; "
                "it is not confirmatory evidence."
            ),
        }
        write_csv(
            OUT / "metrics_summary.csv", stage6_summary_rows
        )
        write_csv(
            OUT / "method_contrasts.csv", stage6_contrast_rows
        )
        write_csv(
            OUT / "pose_error_noninferiority.csv",
            stage6_pose_ratio_rows,
        )
        write_json(
            OUT / "stage6_development_decision.json",
            stage6_decision,
        )
        print(json.dumps(stage6_decision, indent=2))
    except Exception:
        record_failure("stage6_development_analysis")


In [ ]:
# Phase E — development plots.


def make_stage6_plots(summary_rows, contrast_rows):
    development = [
        row
        for row in summary_rows
        if row["split"] == DEVELOPMENT_SPLIT
    ]
    colors = {
        method: plt.cm.tab10(index)
        for index, method in enumerate(METHODS)
    }
    figure, axes = plt.subplots(
        2, 2, figsize=(16, 10), constrained_layout=True
    )
    for row_index, environment in enumerate(ENVIRONMENT):
        selected = [
            row
            for row in development
            if row["environment"] == environment
        ]
        x = np.arange(len(METHODS))
        by_method = {row["method"]: row for row in selected}
        regret = [
            by_method[method]["normalized_regret"]
            for method in METHODS
        ]
        ranking = [
            by_method[method]["weighted_pairwise_accuracy"]
            for method in METHODS
        ]
        axes[row_index, 0].bar(
            x,
            regret,
            color=[colors[method] for method in METHODS],
        )
        axes[row_index, 1].bar(
            x,
            ranking,
            color=[colors[method] for method in METHODS],
        )
        axes[row_index, 0].set_title(
            f"{environment}: normalized regret (lower is better)"
        )
        axes[row_index, 1].set_title(
            f"{environment}: weighted ranking (higher is better)"
        )
        for axis in axes[row_index]:
            axis.set_xticks(x)
            axis.set_xticklabels(
                METHODS, rotation=30, ha="right", fontsize=8
            )
            axis.grid(axis="y", alpha=0.25)
    figure.savefig(
        PLOT_DIR / "planning_by_method.png", dpi=180
    )
    plt.close(figure)

    figure, axes = plt.subplots(
        1, 2, figsize=(15, 5), constrained_layout=True
    )
    for index, environment in enumerate(ENVIRONMENT):
        selected = [
            row
            for row in development
            if row["environment"] == environment
        ]
        by_method = {row["method"]: row for row in selected}
        values = [
            by_method[method]["pose_error"] for method in METHODS
        ]
        axes[index].bar(
            np.arange(len(METHODS)),
            values,
            color=[colors[method] for method in METHODS],
        )
        axes[index].set_title(
            f"{environment}: physical pose error"
        )
        axes[index].set_xticks(np.arange(len(METHODS)))
        axes[index].set_xticklabels(
            METHODS, rotation=30, ha="right", fontsize=8
        )
        axes[index].grid(axis="y", alpha=0.25)
    figure.savefig(
        PLOT_DIR / "pose_error_by_method.png", dpi=180
    )
    plt.close(figure)

    primary = [
        row
        for row in contrast_rows
        if row["metric"]
        in ["normalized_regret", "weighted_pairwise_accuracy"]
    ]
    labels = [
        f"{row['environment']} | {row['comparator']} | "
        f"{row['metric']}"
        for row in primary
    ]
    estimates = np.asarray(
        [row["estimate"] for row in primary]
    )
    low = np.asarray([row["low"] for row in primary])
    high = np.asarray([row["high"] for row in primary])
    y = np.arange(len(primary))
    figure, axis = plt.subplots(
        figsize=(14, max(7, 0.42 * len(primary))),
        constrained_layout=True,
    )
    axis.errorbar(
        estimates,
        y,
        xerr=np.vstack([estimates - low, high - estimates]),
        fmt="o",
        color="#159957",
        ecolor="#4f5b6a",
        capsize=3,
    )
    axis.axvline(0.0, color="black", linewidth=1)
    axis.set_yticks(y)
    axis.set_yticklabels(labels, fontsize=8)
    axis.set_xlabel(
        "positive means counterfactual action-effect adapter is better"
    )
    axis.set_title(
        "State-clustered Stage 6 development contrasts"
    )
    figure.savefig(
        PLOT_DIR / "method_contrasts.png", dpi=180
    )
    plt.close(figure)


if not PIPELINE_FAILED:
    try:
        make_stage6_plots(
            stage6_summary_rows, stage6_contrast_rows
        )
    except Exception:
        record_failure("stage6_plotting")


In [ ]:
# Phase F — compact result packaging and automatic download.


def package_stage6_results():
    files = []
    for path in sorted(OUT.rglob("*")):
        if not path.is_file():
            continue
        if path == RESULT_ZIP:
            continue
        relative = path.relative_to(OUT)
        if relative.parts and relative.parts[0] == "intermediate":
            continue
        files.append(path)
    manifest = {
        "bundle": RESULT_ZIP.name,
        "evidence_status": EVIDENCE_STATUS,
        "pipeline_failed": bool(PIPELINE_FAILED),
        "file_count": int(len(files)),
        "files": [
            {
                "path": path.relative_to(OUT).as_posix(),
                "bytes": int(path.stat().st_size),
                "sha256": sha256_file(path),
            }
            for path in files
        ],
    }
    write_json(OUT / "result_zip_manifest.json", manifest)
    files.append(OUT / "result_zip_manifest.json")
    with zipfile.ZipFile(
        RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        for path in files:
            archive.write(
                path, arcname=path.relative_to(OUT).as_posix()
            )
    return manifest


RESULT_ZIP = OUT / "stage6_result_bundle.zip"
try:
    stage6_bundle_manifest = package_stage6_results()
    print(json.dumps(stage6_bundle_manifest, indent=2))
    print(
        "RUN_STATUS:",
        "FAILED" if PIPELINE_FAILED else "SUCCESS",
    )
    print("RESULT_ZIP:", RESULT_ZIP)
    if DOWNLOAD_RESULTS:
        from google.colab import files

        files.download(str(RESULT_ZIP))
except Exception:
    record_failure("stage6_result_packaging")
    raise
